# Analisis tahap 6: uji statistik dan tabel Bab 4

Menjalankan `thesis/analysis.py` atas seluruh hasil eksperimen utama. **Cukup CPU** (Accelerator: None), jadi tidak memakai kuota GPU.

**Keluaran** (`analisis.zip`):
- `laporan.md`: tabel-tabel utama.
- `runs.csv`: metrik per run dan per periode uji.
- `summary.csv`: rerata dan SD per konfigurasi.
- `tests.csv`: uji Wilcoxon berpasangan dengan koreksi Holm.
- `patterns.csv`: recall per pola pencucian uang.
- `cost.csv`: biaya komputasi.
- `pr_*.png`: kurva PR rerata.

**Langkah:**
1. Di laptop, buat bundel hasil:
   `powershell -ExecutionPolicy Bypass -File thesis\pack_results.ps1 -Root "<...>\project_aml\hasil_kaggle" -Out "<...>\project_aml\hasil_tesis_bundle.zip"`
   Tambahkan `-IncludeVal` bila analisis ambang dari data validasi dipakai.
2. Kaggle -> **Datasets -> New Dataset**: unggah `hasil_tesis_bundle.zip` (Private). Untuk versi berikutnya: **New Version** pada dataset yang sama.
3. Notebook ini: **Add Input** -> dataset tersebut; **Accelerator: None**; **Internet: On**.
4. **Run All**, lalu unduh `analisis.zip` dari Output.

In [ ]:
VAL_THRESHOLD = False   # True: F1 dengan ambang terbaik dari data validasi (bundel harus dibuat dengan -IncludeVal)

In [ ]:
import glob, os, subprocess, zipfile

REPO_URL = "https://github.com/aanpujianto070990/MEGA-GNN-TMLR.git"
REPO_DIR = "/kaggle/working/MEGA-GNN-TMLR"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

# Folder bundel: Kaggle biasanya mengekstrak zip yang diunggah; kalau belum, ekstrak di sini.
manifests = glob.glob("/kaggle/input/**/bundle_manifest.csv", recursive=True)
if not manifests:
    zips = glob.glob("/kaggle/input/**/hasil_tesis_bundle*.zip", recursive=True)
    assert zips, "Bundel tidak ditemukan -- tambahkan dataset hasil_tesis_bundle sebagai Input"
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall("/kaggle/working/bundle")
    manifests = ["/kaggle/working/bundle/bundle_manifest.csv"]
BUNDLE = os.path.dirname(manifests[0])
print("Bundel:", BUNDLE)
print(open(manifests[0]).read().count("\n") - 1, "run di manifest")

In [ ]:
OUT = "/kaggle/working/analisis"
cmd = ["python", f"{REPO_DIR}/thesis/analysis.py", "--results", f"{BUNDLE}/runs",
       "--meta-dir", f"{BUNDLE}/meta", "--out", OUT]
if VAL_THRESHOLD:
    cmd.append("--val-threshold")
subprocess.run(cmd, check=True)

In [ ]:
from IPython.display import Markdown, Image, display

display(Markdown(open(f"{OUT}/laporan.md", encoding="utf-8").read()))
for png in sorted(glob.glob(f"{OUT}/pr_*.png")):
    display(Image(png))

In [ ]:
with zipfile.ZipFile("/kaggle/working/analisis.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for path in sorted(glob.glob(f"{OUT}/*")):
        z.write(path, os.path.join("analisis", os.path.basename(path)))
print("Unduh /kaggle/working/analisis.zip dari Output")